# S&P 500 market breadth: one month, one year, and the path between

This notebook uses the Zibaldon/Tyche Data API for S&P 500 constituents and daily prices. It tracks familiar breadth measures—advance/decline, moving-average participation, new highs/lows, and the share of stocks with positive returns—alongside two deliberately simple diagnostics:

- **Breadth impulse:** the 21-trading-day change in the share of stocks with positive one-month returns.
- **Concentration gap:** SPY's return minus the median constituent return; a large positive gap means the cap-weighted index is outrunning the typical stock.

Historical membership comes from fja05680's [`sp500_ticker_start_end.csv`](https://github.com/fja05680/sp500), which reconstructs point-in-time S&P 500 membership since 1996 and preserves repeated membership spells. Set `SP500_MEMBERSHIP_SOURCE` to a local snapshot of the same file when reproducibility or offline execution matters.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from dotenv import load_dotenv

from tyche_client import TycheClient

load_dotenv()
client = TycheClient(os.getenv("TYCHE_BASE_URL"), os.getenv("TYCHE_API_KEY"))
plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.2f}".format

## Configuration and membership history

Ten years is long enough to show several market regimes while keeping the download practical. Change `START` freely. In the source table, `start_date` is inclusive and `end_date` is the first date absent from the index, so the notebook treats it as exclusive.

In [ ]:
START = "2015-01-01"
MEMBERSHIP_SOURCE = os.getenv(
    "SP500_MEMBERSHIP_SOURCE",
    "https://raw.githubusercontent.com/fja05680/sp500/refs/heads/master/sp500_ticker_start_end.csv",
)

current = client.constituents("sp500")
if current.empty or "ticker" not in current:
    raise RuntimeError("The API returned no S&P 500 constituents.")
current["ticker"] = current["ticker"].str.upper()

membership = pd.read_csv(MEMBERSHIP_SOURCE).rename(columns={"start_date": "start", "end_date": "end"})
required = {"ticker", "start", "end"}
if not required.issubset(membership.columns):
    raise ValueError(f"Membership source must contain {sorted(required)}")
membership = membership[["ticker", "start", "end"]].copy()
membership["ticker"] = membership["ticker"].str.upper()
membership[["start", "end"]] = membership[["start", "end"]].apply(pd.to_datetime)
if membership["start"].isna().any() or (membership["end"] < membership["start"]).any():
    raise ValueError("Membership source contains invalid intervals")

# Match class-share punctuation to the API when its current list uses a hyphen.
api_tickers = set(current["ticker"])
membership["ticker"] = membership["ticker"].map(
    lambda ticker: ticker.replace(".", "-") if ticker.replace(".", "-") in api_tickers else ticker
)
membership_mode = f"point-in-time membership: {MEMBERSHIP_SOURCE}"

tickers = sorted(set(current["ticker"]) | set(membership.loc[membership["end"].isna() | (membership["end"] > START), "ticker"]))
print(f"{len(tickers):,} unique tickers | {membership_mode}")

## Retrieve prices from Zibaldon

The client batches the constituent request automatically. SPY is fetched separately as the cap-weighted market benchmark. Delisted names absent from the price service remain missing; the coverage chart below makes that limitation visible.

In [ ]:
prices = client.price_matrix(tickers, start=START).sort_index()
spy = client.prices("SPY", start=START)["Close"].rename("SPY")
if prices.empty or spy.empty:
    raise RuntimeError("No price history returned; check the API URL, key, and date range.")

prices = prices.loc[: spy.index.max()]
print(f"{prices.index.min().date()} to {prices.index.max().date()} | {prices.shape[1]:,} price series")

## Apply point-in-time membership

A stock contributes only while it belongs to the index. A return that crosses an entry or exit date is also excluded. The tiny assertion is the notebook's check that interval boundaries behave as intended.

In [ ]:
def membership_mask(index, columns, periods):
    mask = pd.DataFrame(False, index=index, columns=columns)
    for row in periods.itertuples(index=False):
        if row.ticker in mask:
            end = row.end if pd.notna(row.end) else index.max() + pd.Timedelta(days=1)
            mask.loc[(index >= row.start) & (index < end), row.ticker] = True
    return mask

_days = pd.date_range("2024-01-01", periods=3)
_period = pd.DataFrame({"ticker": ["A"], "start": [_days[1]], "end": [_days[2]]})
assert membership_mask(_days, ["A"], _period)["A"].tolist() == [False, True, False]

member = membership_mask(prices.index, prices.columns, membership)
active_prices = prices.where(member)
member_returns = prices.pct_change(fill_method=None).where(member & member.shift(1, fill_value=False))

## Calculate breadth

All shares are equal-weighted across eligible members with enough data on each date. One month is 21 trading days and one year is 252. The participation score is a transparent 0–100 average of four components; it is a summary gauge, not a trading rule.

In [ ]:
ret_1m = prices.pct_change(21, fill_method=None).where(member & member.shift(21, fill_value=False))
ret_1y = prices.pct_change(252, fill_method=None).where(member & member.shift(252, fill_value=False))
ma_50, ma_200 = prices.rolling(50).mean(), prices.rolling(200).mean()
high_252, low_252 = prices.rolling(252).max(), prices.rolling(252).min()

breadth = pd.DataFrame(index=prices.index)
breadth["positive_1d"] = (member_returns > 0).where(member_returns.notna()).mean(axis=1)
breadth["positive_1m"] = (ret_1m > 0).where(ret_1m.notna()).mean(axis=1)
breadth["positive_1y"] = (ret_1y > 0).where(ret_1y.notna()).mean(axis=1)
breadth["above_50dma"] = (prices > ma_50).where(member & ma_50.notna()).mean(axis=1)
breadth["above_200dma"] = (prices > ma_200).where(member & ma_200.notna()).mean(axis=1)
breadth["new_high_1y"] = (prices >= high_252).where(member & high_252.notna()).mean(axis=1)
breadth["new_low_1y"] = (prices <= low_252).where(member & low_252.notna()).mean(axis=1)
breadth["high_low_net"] = breadth["new_high_1y"] - breadth["new_low_1y"]
breadth["ad_line"] = np.sign(member_returns).sum(axis=1).cumsum()
breadth["median_1m"] = ret_1m.median(axis=1)
breadth["median_1y"] = ret_1y.median(axis=1)
breadth["dispersion_1m"] = ret_1m.quantile(.75, axis=1) - ret_1m.quantile(.25, axis=1)
breadth["breadth_impulse"] = breadth["positive_1m"].diff(21)
breadth["coverage"] = active_prices.notna().sum(axis=1).div(member.sum(axis=1).replace(0, np.nan))
breadth["members"] = member.sum(axis=1)
breadth["participation_score"] = 100 * pd.concat([
    breadth["positive_1m"], breadth["above_50dma"], breadth["above_200dma"],
    (breadth["high_low_net"] + 1) / 2,
], axis=1).mean(axis=1)

spy_1m, spy_1y = spy.pct_change(21), spy.pct_change(252)
breadth["concentration_gap_1m"] = spy_1m.reindex(breadth.index) - breadth["median_1m"]
breadth["concentration_gap_1y"] = spy_1y.reindex(breadth.index) - breadth["median_1y"]
breadth = breadth.loc[breadth["coverage"].notna()]
breadth.tail(3)

## Evolution dashboard

The 50% line separates broad participation from minority participation. A rising SPY with weakening one-month and one-year breadth is a useful divergence to investigate, not by itself a sell signal.

In [ ]:
view = breadth.index >= pd.Timestamp(START)
fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True, constrained_layout=True)

(spy / spy.iloc[0] * 100).plot(ax=axes[0], color="#172554", lw=1.8)
axes[0].set(title="SPY (start = 100)", ylabel="Index")

breadth.loc[view, ["positive_1m", "positive_1y"]].plot(ax=axes[1], color=["#2563eb", "#7c3aed"], lw=1.4)
axes[1].axhline(.5, color="0.3", ls="--", lw=.8)
axes[1].set(title="Stocks with positive trailing returns", ylabel="Share")
axes[1].yaxis.set_major_formatter(PercentFormatter(1))

breadth.loc[view, ["above_50dma", "above_200dma"]].plot(ax=axes[2], color=["#0891b2", "#ea580c"], lw=1.3)
axes[2].axhline(.5, color="0.3", ls="--", lw=.8)
axes[2].set(title="Moving-average participation", ylabel="Share")
axes[2].yaxis.set_major_formatter(PercentFormatter(1))

breadth.loc[view, "participation_score"].plot(ax=axes[3], color="#059669", lw=1.4, label="participation score")
axes[3].axhline(50, color="0.3", ls="--", lw=.8)
axes[3].set(title="Composite participation and 52-week highs minus lows", ylabel="Score")
right = axes[3].twinx()
breadth.loc[view, "high_low_net"].plot(ax=right, color="#dc2626", alpha=.45, lw=.9, label="high-low net")
right.yaxis.set_major_formatter(PercentFormatter(1))
right.set_ylabel("Net share")

breadth.loc[view, "coverage"].plot(ax=axes[4], color="#475569", lw=1.2, label="price coverage")
axes[4].set(title=f"Point-in-time data coverage — {membership_mode}", ylabel="Coverage", xlabel="")
axes[4].yaxis.set_major_formatter(PercentFormatter(1))
axes[4].set_ylim(0, 1.05)
fig.suptitle("S&P 500 market breadth evolution", fontsize=18, fontweight="bold")
plt.show()

## Current one-month versus one-year cross-section

Each point is a current constituent. The quadrants separate recent leadership, recent reversals, persistent weakness, and persistent strength. Sector colors help show whether breadth is broad or clustered. Labels identify only the largest absolute one-month moves to keep the chart readable.

In [ ]:
as_of = ret_1m.dropna(how="all").index[-1]
latest = pd.DataFrame({"return_1m": ret_1m.loc[as_of], "return_1y": ret_1y.loc[as_of]}).dropna()
meta_cols = [c for c in ["ticker", "sector"] if c in current]
latest = latest.join(current[meta_cols].drop_duplicates("ticker").set_index("ticker"))
latest["sector"] = latest.get("sector", pd.Series(index=latest.index, dtype=object)).fillna("Unclassified")
colors = dict(zip(sorted(latest["sector"].unique()), plt.cm.tab20.colors))

fig, ax = plt.subplots(figsize=(14, 9))
for sector, group in latest.groupby("sector"):
    ax.scatter(group["return_1m"], group["return_1y"], s=34, alpha=.72, color=colors[sector], label=f"{sector} ({len(group)})")
for ticker, row in latest.loc[latest["return_1m"].abs().nlargest(12).index].iterrows():
    ax.annotate(ticker, (row["return_1m"], row["return_1y"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.axhline(0, color="0.25", lw=.9)
ax.axvline(0, color="0.25", lw=.9)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set(title=f"Current S&P 500 return breadth — {as_of.date()}", xlabel="Past month return", ylabel="Past year return")
ax.legend(ncol=2, fontsize=8, frameon=True, loc="best")
plt.show()

## Is the index representative of the typical stock?

The concentration gap directly compares SPY with the median member. Positive values indicate that larger companies are leading; negative values indicate that the typical stock is doing better than the cap-weighted index. Breadth impulse captures whether one-month participation is improving quickly.

In [ ]:
recent = breadth.tail(3 * 252)
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, constrained_layout=True)
comparisons = [
    (spy_1m, breadth["median_1m"], "One-month return: SPY vs median stock"),
    (spy_1y, breadth["median_1y"], "One-year return: SPY vs median stock"),
]
for ax, (market, median, title) in zip(axes[0], comparisons):
    pd.concat([market.rename("SPY"), median.rename("median member")], axis=1).loc[recent.index].plot(ax=ax, lw=1.2)
    ax.axhline(0, color="0.3", lw=.8)
    ax.set_title(title)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
recent[["concentration_gap_1m", "concentration_gap_1y"]].plot(ax=axes[1, 0], lw=1.1)
axes[1, 0].axhline(0, color="0.3", lw=.8)
axes[1, 0].set_title("Concentration gap")
axes[1, 0].yaxis.set_major_formatter(PercentFormatter(1))
recent["breadth_impulse"].plot(ax=axes[1, 1], color="#0284c7", lw=1.1)
axes[1, 1].axhline(0, color="0.3", lw=.8)
axes[1, 1].set_title("Breadth impulse: 21-day change in one-month participation")
axes[1, 1].yaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle("Breadth divergences — latest three years", fontsize=16, fontweight="bold")
plt.show()

## Latest breadth snapshot

The percentile column places today's reading within this notebook's available history. Very low participation plus a negative breadth impulse describes broad deterioration; a strong SPY return paired with a large concentration gap describes narrow leadership.

In [ ]:
labels = {
    "positive_1m": "Stocks positive over 1 month",
    "positive_1y": "Stocks positive over 1 year",
    "above_50dma": "Stocks above 50-day average",
    "above_200dma": "Stocks above 200-day average",
    "high_low_net": "52-week highs minus lows",
    "breadth_impulse": "Breadth impulse",
    "concentration_gap_1m": "SPY minus median stock, 1 month",
    "concentration_gap_1y": "SPY minus median stock, 1 year",
    "participation_score": "Participation score (0–100)",
    "coverage": "Point-in-time price coverage",
}
snapshot = breadth[list(labels)].iloc[-1].rename("latest").to_frame()
snapshot["historical_percentile"] = [breadth[col].rank(pct=True).iloc[-1] for col in labels]
snapshot.index = snapshot.index.map(labels)
percent_rows = snapshot.index.drop("Participation score (0–100)")
display(snapshot.style.format("{:.1%}", subset=pd.IndexSlice[percent_rows, ["latest"]])
        .format("{:.1f}", subset=pd.IndexSlice[["Participation score (0–100)"], ["latest"]])
        .format("{:.0%}", subset=pd.IndexSlice[:, ["historical_percentile"]])
        .background_gradient(subset=["historical_percentile"], cmap="RdYlGn", vmin=0, vmax=1))

print(f"As of {breadth.index[-1].date()} | {membership_mode}")